# Notebook 05: SQL Pipeline & Data Warehouse Construction

## Overview

This notebook represents the transition from engineered datasets to a production-style relational data warehouse.

The primary objective is to transform the curated IPL datasets produced in previous notebooks into a fully normalized Star Schema, persist the warehouse inside MySQL, and validate the integrity of the database for downstream analytics.

Unlike the previous notebooks, this notebook performs **no feature engineering or business transformations**. Instead, it orchestrates reusable modules implemented inside the `src` package to build, validate, and persist the warehouse.

---

## Objectives

- Load engineered match and delivery datasets.
- Construct warehouse dimension tables.
- Assign surrogate keys.
- Build fact tables.
- Validate Star Schema integrity.
- Create the relational warehouse in MySQL.
- Load warehouse tables into the database.
- Execute SQL validation queries.
- Generate warehouse loading and validation summaries.

---

## Input

The notebook consumes the engineered datasets generated during Notebook 04.

- Processed Match Dataset
- Processed Delivery Dataset

---

## Output

A fully populated MySQL data warehouse consisting of:

### Dimensions

- `dim_season`
- `dim_team`
- `dim_player`
- `dim_venue`
- `dim_match`

### Facts

- `fact_match`
- `fact_player_match`
- `fact_delivery`

The resulting warehouse serves as the primary analytical data source for:

- SQL Analytics
- Business Intelligence (Power BI)
- Machine Learning
- Advanced Cricket Analytics

---

## Notebook Philosophy

This notebook follows the project-wide engineering principles:

- Thin notebook orchestration
- Reusable business logic inside `src`
- Modular architecture
- Production-quality implementation
- Interview-ready engineering practices

In [1]:
# ============================================================
# Standard Library Imports
# ============================================================

from pathlib import Path
from datetime import datetime
import warnings
import time

# ============================================================
# Third-Party Imports
# ============================================================

import pandas as pd

# ============================================================
# Project Configuration
# ============================================================

from config.config import *

# ============================================================
# Workflow Imports
# ============================================================

from src.workflows.database_pipeline import run_database_pipeline

# ============================================================
# Display Configuration
# ============================================================

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# ============================================================
# Notebook Metadata
# ============================================================

NOTEBOOK_NAME = "Notebook 05 - SQL Pipeline"

PIPELINE_START_TIME = time.perf_counter()

print("=" * 70)
print(NOTEBOOK_NAME)
print("=" * 70)
print(f"Execution Started : {datetime.now():%Y-%m-%d %H:%M:%S}")
print("=" * 70)

Notebook 05 - SQL Pipeline
Execution Started : 2026-07-04 19:06:39


## Environment Validation

Before executing the SQL pipeline, verify that the project configuration,
required directories, engineered datasets, and database settings are available.

This prevents unnecessary failures during warehouse construction and database loading.

In [2]:
# ============================================================
# Environment Validation
# ============================================================

print("\nEnvironment Validation")
print("-" * 80)

validation_checks = {
    "Processed Data Directory": PROCESSED_DATA_DIR.exists(),
    "Warehouse Directory": WAREHOUSE_DIR.exists(),
    "Analytics Directory": ANALYTICS_DATA_DIR.exists(),
}

for item, status in validation_checks.items():
    symbol = "✓" if status else "✗"
    print(f"{symbol} {item}")

if all(validation_checks.values()):
    print("\n✓ Environment validation completed successfully.")
else:
    raise FileNotFoundError(
        "One or more required project directories are missing."
    )


Environment Validation
--------------------------------------------------------------------------------
✓ Processed Data Directory
✓ Warehouse Directory
✓ Analytics Directory

✓ Environment validation completed successfully.


## Execute SQL Pipeline

Run the complete database persistence workflow.

The workflow performs the following operations:

1. Load engineered datasets
2. Build warehouse dimensions
3. Assign surrogate keys
4. Construct fact tables
5. Validate Star Schema
6. Create MySQL schema
7. Load warehouse tables
8. Execute SQL validation
9. Return execution summary

In [3]:
# ============================================================
# Execute Database Pipeline
# ============================================================

print("\nExecuting SQL Pipeline...")
print("-" * 80)

try:
    matches_path = str(STAGING_DIR / "match_features.parquet")
    deliveries_path = str(STAGING_DIR / "deliveries_processed.parquet")
    pipeline_results = run_database_pipeline(
        matches_path=matches_path,
        deliveries_path=deliveries_path,
    )

    print("\n✓ SQL Pipeline completed successfully.")

except Exception as error:
    print("\n✗ SQL Pipeline failed.")
    raise error


Executing SQL Pipeline...
--------------------------------------------------------------------------------


Dropping database 'ipl_analytics'.



✓ SQL Pipeline completed successfully.


## Pipeline Execution Summary

Summarize the overall execution status of the SQL pipeline.

This section provides a concise overview of the warehouse construction,
database loading, and overall execution time.

In [4]:
# ============================================================
# Pipeline Execution Summary
# ============================================================

print("\n")
print("=" * 80)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 80)

print(f"Pipeline Name      : {NOTEBOOK_NAME}")
print(f"Execution Status   : SUCCESS")
print(f"Execution Time     : {pipeline_results.execution_time:.2f} seconds")
print(f"Completed At       : {datetime.now():%d-%b-%Y %I:%M:%S %p}")

print("=" * 80)



PIPELINE EXECUTION SUMMARY
Pipeline Name      : Notebook 05 - SQL Pipeline
Execution Status   : SUCCESS
Execution Time     : 61.98 seconds
Completed At       : 04-Jul-2026 07:07:41 PM


## Pipeline Output Inspection

Inspect the workflow output returned by the database pipeline.

This step is useful during development and debugging to understand
the structure of the returned execution object before generating
formatted reports.

In [5]:
print("\nPipeline Output Type")
print("-" * 80)
print(type(pipeline_results))

print("\nPipeline Status")
print("-" * 80)

print(f"Status          : {pipeline_results.status}")
print(f"Database        : {pipeline_results.database_name}")
print(f"Execution Time  : {pipeline_results.execution_time:.2f} sec")

print("\nLoad Summary")
print("-" * 80)

for table, rows in pipeline_results.load_summary.items():
    print(f"{table:<25}{rows:>10,}")

print("\nValidation Summary")
print("-" * 80)

print(pipeline_results.validation_summary)


Pipeline Output Type
--------------------------------------------------------------------------------
<class 'src.workflows.database_pipeline.DatabasePipelineResult'>

Pipeline Status
--------------------------------------------------------------------------------
Status          : SUCCESS
Database        : ipl_analytics
Execution Time  : 61.98 sec

Load Summary
--------------------------------------------------------------------------------
dim_season                       18
dim_team                         17
dim_player                      818
dim_venue                        46
dim_match                     1,243
fact_match                    1,243
fact_player_match            18,768
fact_delivery               295,732

Validation Summary
--------------------------------------------------------------------------------
{'dimensions': {'dim_season': {'primary_key': 'season_key', 'duplicate_keys': 0, 'null_keys': 0, 'is_valid': True}, 'dim_team': {'primary_key': 'team_key', 'duplica

## Warehouse Construction Status

The warehouse construction workflow has completed successfully.

At this stage:

- Star Schema has been generated.
- Relational schema has been created.
- Warehouse tables have been loaded.
- Database validation has completed.

The warehouse is now ready for analytical SQL queries,
Power BI dashboards, and downstream machine learning workflows.

In [6]:
# ============================================================
# Warehouse Status
# ============================================================

print("\n")
print("=" * 80)
print("WAREHOUSE STATUS")
print("=" * 80)

status_messages = [
    "Star Schema Constructed",
    "Dimension Tables Created",
    "Fact Tables Created",
    "MySQL Schema Created",
    "Warehouse Loaded",
    "Database Validation Completed",
]

for message in status_messages:
    print(f"✓ {message}")

print("=" * 80)



WAREHOUSE STATUS
✓ Star Schema Constructed
✓ Dimension Tables Created
✓ Fact Tables Created
✓ MySQL Schema Created
✓ Warehouse Loaded
✓ Database Validation Completed


## Warehouse Load Summary

The SQL pipeline has successfully completed the warehouse construction and
database loading process.

The following summary presents the number of records loaded into each
dimension and fact table.

In [7]:
# ============================================================
# Warehouse Load Summary
# ============================================================

print("\n")
print("=" * 80)
print("WAREHOUSE LOAD SUMMARY")
print("=" * 80)

print("\nDimensions")
print("-" * 80)

dimension_tables = [
    "dim_season",
    "dim_team",
    "dim_player",
    "dim_venue",
    "dim_match",
]

for table in dimension_tables:
    print(f"{table:<25}{pipeline_results.load_summary.get(table, 0):>10,}")

print("\nFact Tables")
print("-" * 80)

fact_tables = [
    "fact_match",
    "fact_player_match",
    "fact_delivery",
]

for table in fact_tables:
    print(f"{table:<25}{pipeline_results.load_summary.get(table, 0):>10,}")

print("\nExecution")
print("-" * 80)

print(f"{'Status':<25}SUCCESS")
print(f"{'Execution Time':<25}{pipeline_results.execution_time:.2f} seconds")

print("=" * 80)



WAREHOUSE LOAD SUMMARY

Dimensions
--------------------------------------------------------------------------------
dim_season                       18
dim_team                         17
dim_player                      818
dim_venue                        46
dim_match                     1,243

Fact Tables
--------------------------------------------------------------------------------
fact_match                    1,243
fact_player_match            18,768
fact_delivery               295,732

Execution
--------------------------------------------------------------------------------
Status                   SUCCESS
Execution Time           61.98 seconds


## Database Load Statistics

Compute high-level warehouse statistics from the completed loading process.

These statistics provide a quick overview of the size of the analytical
warehouse.

In [8]:
# ============================================================
# Warehouse Statistics
# ============================================================

total_dimensions = sum(
    pipeline_results.load_summary[table]
    for table in dimension_tables
)

total_facts = sum(
    pipeline_results.load_summary[table]
    for table in fact_tables
)

total_rows = total_dimensions + total_facts

print("\n")
print("=" * 80)
print("WAREHOUSE STATISTICS")
print("=" * 80)

print(f"{'Dimension Tables':<30}{len(dimension_tables)}")
print(f"{'Fact Tables':<30}{len(fact_tables)}")
print(f"{'Dimension Records':<30}{total_dimensions:,}")
print(f"{'Fact Records':<30}{total_facts:,}")
print(f"{'Total Warehouse Records':<30}{total_rows:,}")

print("=" * 80)



WAREHOUSE STATISTICS
Dimension Tables              5
Fact Tables                   3
Dimension Records             2,142
Fact Records                  315,743
Total Warehouse Records       317,885


## Pipeline Validation Summary

The following checklist confirms the successful completion of every major
phase of the SQL pipeline.

In [9]:
# ============================================================
# Pipeline Validation Summary
# ============================================================

validation_steps = [
    "Engineered datasets loaded",
    "Dimensions constructed",
    "Surrogate keys assigned",
    "Fact tables generated",
    "Star schema validated",
    "Database schema created",
    "Warehouse loaded into MySQL",
    "Pipeline completed successfully",
]

print("\n")
print("=" * 80)
print("PIPELINE VALIDATION SUMMARY")
print("=" * 80)

for step in validation_steps:
    print(f"✓ {step}")

print("=" * 80)



PIPELINE VALIDATION SUMMARY
✓ Engineered datasets loaded
✓ Dimensions constructed
✓ Surrogate keys assigned
✓ Fact tables generated
✓ Star schema validated
✓ Database schema created
✓ Warehouse loaded into MySQL
✓ Pipeline completed successfully


## Database Verification

Verify the integrity of the MySQL warehouse by executing SQL validation
queries against the persisted tables.

The following checks are performed:

- Row count verification
- Duplicate primary/composite key validation

This confirms that the warehouse has been loaded successfully and that
the primary keys remain unique after persistence.

In [10]:
# ============================================================
# Database Verification
# ============================================================

from src.database import (
    create_connection,
    close_connection,
    get_row_count_query,
    get_duplicate_keys_query,
)

print("\n")
print("=" * 80)
print("DATABASE VERIFICATION")
print("=" * 80)

connection = create_connection()
cursor = connection.cursor()

validation_tables = {
    "dim_season": ["season_key"],
    "dim_team": ["team_key"],
    "dim_player": ["player_key"],
    "dim_venue": ["venue_key"],
    "dim_match": ["match_key"],
    "fact_match": ["match_key"],
    "fact_player_match": ["match_key", "player_key"],
    "fact_delivery": [
        "match_key",
        "innings",
        "over_number",
        "ball_number",
    ],
}

print(f"{'Table':<25}{'Rows':>12}{'Duplicates':>18}")
print("-" * 60)

try:

    for table_name, primary_keys in validation_tables.items():

        # ----------------------------------------------------
        # Row Count
        # ----------------------------------------------------

        cursor.execute(
            get_row_count_query(table_name)
        )

        row_count = cursor.fetchone()[0]

        # ----------------------------------------------------
        # Duplicate Keys
        # ----------------------------------------------------

        cursor.execute(
            get_duplicate_keys_query(
                table_name=table_name,
                key_columns=primary_keys,
            )
        )

        duplicate_rows = cursor.fetchall()

        duplicate_count = len(duplicate_rows)

        print(
            f"{table_name:<25}"
            f"{row_count:>12,}"
            f"{duplicate_count:>18}"
        )

finally:

    cursor.close()
    close_connection(connection)

print("=" * 60)
print("✓ Database validation completed successfully.")
print("=" * 80)



DATABASE VERIFICATION
Table                            Rows        Duplicates
------------------------------------------------------------
dim_season                         18                 0
dim_team                           17                 0
dim_player                        818                 0
dim_venue                          46                 0
dim_match                       1,243                 0
fact_match                      1,243                 0
fact_player_match              18,768                 0


fact_delivery                 295,732                 0
✓ Database validation completed successfully.


# Conclusion

Notebook 05 successfully transformed the engineered IPL datasets into a
production-grade relational data warehouse.

The notebook orchestrated the complete warehouse construction process,
including:

- Building conformed dimensions
- Assigning surrogate keys
- Constructing analytical fact tables
- Validating the Star Schema
- Creating the MySQL schema
- Bulk loading warehouse tables
- Verifying database integrity using SQL validation queries

The completed warehouse now serves as the single source of truth for all
subsequent analytical workloads, including SQL analytics, Power BI dashboards,
and predictive modeling.

With the completion of this notebook, the project transitions from
**Data Engineering** to **Analytics Engineering**, where the relational
warehouse becomes the primary data source for business insights.

In [11]:
# ============================================================
# Notebook Completion
# ============================================================

NOTEBOOK_END_TIME = time.perf_counter()

TOTAL_NOTEBOOK_TIME = NOTEBOOK_END_TIME - PIPELINE_START_TIME

print("\n")
print("=" * 80)
print("NOTEBOOK 05 COMPLETED SUCCESSFULLY")
print("=" * 80)

print(f"{'Notebook':<25}: {NOTEBOOK_NAME}")
print(f"{'Execution Status':<25}: SUCCESS")
print(f"{'Database':<25}: ipl_analytics")
print(f"{'Dimension Tables':<25}: 5")
print(f"{'Fact Tables':<25}: 3")
print(f"{'Total Warehouse Rows':<25}: {total_rows:,}")
print(f"{'Execution Time':<25}: {TOTAL_NOTEBOOK_TIME:.2f} seconds")
print(f"{'Completed At':<25}: {datetime.now():%d-%b-%Y %I:%M:%S %p}")

print("=" * 80)
print("The IPL Analytics Warehouse is ready for SQL Analytics & Power BI.")
print("=" * 80)



NOTEBOOK 05 COMPLETED SUCCESSFULLY
Notebook                 : Notebook 05 - SQL Pipeline
Execution Status         : SUCCESS
Database                 : ipl_analytics
Dimension Tables         : 5
Fact Tables              : 3
Total Warehouse Rows     : 317,885
Execution Time           : 62.63 seconds
Completed At             : 04-Jul-2026 07:07:42 PM
The IPL Analytics Warehouse is ready for SQL Analytics & Power BI.
